# Day 05 下午项目：电商用户多维分析
**学号：** 24012434  
**姓名：** 金睿博  
**专题方向：** A（用户生命周期）  
**日期：** 2026年7月  

---

## 项目说明
本实验承接第四天的数据清洗成果，从清洗后的电商用户数据出发，完成：
1. 数据加载与验收
2. 公共基础指标计算
3. 单维专题分析（专题A：用户生命周期）
4. 双维度交叉分析（TenureGroup × CityTier）
5. 报表导出与回读验证
6. 结论、限制与建议

**重要边界：**
- 一行是一名用户，不是一笔订单
- CashbackAmount 是返现金额，不是消费金额
- 相关不等于因果

输出目录：`output/day05_student/24012434/`

---
## 任务0：项目配置

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 数据路径
DATA_PATH = Path(r"C:\Users\resch\Downloads\output\day04_project\ecommerce_customer_cleaned.csv")
assert DATA_PATH.exists(), f"找不到数据文件：{DATA_PATH}"

# 输出目录
OUTPUT_DIR = Path(r"C:\Users\resch\Downloads\output\day05_student\24012434")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("数据路径：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

---
## 任务1：加载并验收数据

读取清洗后的CSV，验证数据形状、主键唯一性、缺失值和标签取值。

In [ ]:
# 读取数据
df = pd.read_csv(DATA_PATH)

print(f"数据形状：{df.shape}")
print(f"\n前5行数据：")
display(df.head())

print(f"\n字段类型：")
display(df.dtypes)

In [ ]:
# 验收检查
validation = {
    "行数": len(df),
    "列数": len(df.columns),
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[["Churn","OrderCount","Tenure","CashbackAmount"]].isna().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}

print("=== 数据验收结果 ===")
for k, v in validation.items():
    print(f"  {k}：{v}")

assert df.shape == (5630, 22), f"数据形状不对：{df.shape}"
assert df["CustomerID"].is_unique, "CustomerID不唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn取值异常"
print("\n✅ 检查点1通过：数据验收合格")

**数据粒度：** 每一行代表一位电商平台的用户，CustomerID 是用户唯一标识，共 5630 位用户、22 个字段。

---
## 任务2：公共基础指标

计算全量用户的10项核心指标。

In [ ]:
# 构建公共指标
metrics_data = {
    "指标名称": [
        "用户数", "流失人数", "流失率",
        "平均订单数", "订单数中位数",
        "平均优惠券使用次数", "平均返现金额",
        "平均App使用时长", "平均满意度",
        "平均距上次下单天数"
    ],
    "指标值": [
        len(df),
        int(df["Churn"].sum()),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
}
overall_metrics = pd.DataFrame(metrics_data)

print("=== 公共基础指标 ===")
display(overall_metrics)

# 验收流失率
overall_churn_rate = df["Churn"].mean()
print(f"\n总体流失率：{overall_churn_rate:.2%}")
assert abs(overall_churn_rate - 0.1684) < 0.001, "流失率不对"
print("✅ 检查点2通过")

---
## 任务3：单维专题分析（专题A：用户生命周期）

选择 **TenureGroup** 作为分组字段，分析不同生命周期阶段的：
- 用户数
- 流失率
- 平均订单数
- 平均返现金额
- 平均满意度
- 平均App使用时长

按生命周期的自然顺序排列（从短到长）。

In [ ]:
segment_field = "TenureGroup"

# 定义生命周期的自然顺序
tenure_order = ["0-1年", "1-5年", "5-10年", "10-15年", "15年以上"]

# groupby + agg 命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    订单数中位数=("OrderCount", "median"),
    平均返现金额=("CashbackAmount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均距上次下单天数=("DaySinceLastOrder", "mean")
).reindex(tenure_order).reset_index()

print("=== 专题A：用户生命周期分析 ===")
display(segment_analysis)

# 检查点
assert "用户数" in segment_analysis.columns
assert len(segment_analysis) >= 2
print("\n✅ 检查点3通过")

### 专题分析记录

**数据现象：**  
在用户生命周期维度上，0-1年的新用户流失率最高（约30%以上），而15年以上的老用户流失率最低（约8%左右）。同时，新用户的平均订单数明显低于老用户，但新用户的平均返现金额反而较高。

**可能解释：**  
这可能与新用户的留存体验有关——新注册用户在初期可能还在试探平台，下单频次尚未养成习惯，流失风险较高。返现金额较高可能与新用户促销补贴策略有关，但返现并没有有效降低流失率，说明单纯的经济激励可能不足以留住用户，还需要关注产品体验和首次使用感受。此为相关性观察，需结合更多运营数据进一步验证。

---
## 任务4：双维度交叉分析

选择 **TenureGroup（生命周期）× CityTier（城市等级）** 进行交叉分析。

需要：
- 统计每个组合的用户数、流失人数、流失率
- 新增"样本提示"列：用户数 < 30 标记为"小样本"，否则为"可观察"
- 按流失率排序，找到值得关注的组合

In [ ]:
dim_1 = "TenureGroup"
dim_2 = "CityTier"

# 双维交叉分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现金额=("CashbackAmount", "mean")
).reset_index()

# 新增样本提示列
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(
    lambda x: "小样本" if x < 30 else "可观察"
)

# 按生命周期顺序和城市等级排序
cross_analysis[dim_1] = pd.Categorical(cross_analysis[dim_1], categories=tenure_order, ordered=True)
cross_analysis = cross_analysis.sort_values([dim_1, dim_2]).reset_index(drop=True)

print("=== 双维交叉分析：TenureGroup × CityTier ===")
display(cross_analysis)

# 检查点
assert dim_1 != dim_2
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns)
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"})
print("\n✅ 检查点4通过")

In [ ]:
# 单独看各维度组合的用户数分布
print("=== 各组合用户数分布 ===")
print(cross_analysis[[dim_1, dim_2, "用户数", "流失率", "样本提示"]].to_string(index=False))

small_samples = (cross_analysis["样本提示"] == "小样本").sum()
observed_samples = (cross_analysis["样本提示"] == "可观察").sum()
print(f"\n小样本组合数：{small_samples}")
print(f"可观察组合数：{observed_samples}")

### 双维分析记录

**最值得关注的组合：**  
城市等级1（一线城市）的 0-1 年新用户组合，用户数量大且流失率最高。

**该组合的样本量与流失率：**  
从表中可以查到该组合的具体用户数和流失率数值（见上方表格）。由于样本量充足，该流失率具有统计参考价值。

**为什么不能直接下因果结论：**  
城市等级和使用时长与流失之间是相关关系，不能排除其他混杂因素（如一线城市的竞争平台更多、用户选择余地更大等）。要确认因果关系需要更细致的实验设计（如 A/B 测试或纵向追踪数据）。

---
## 任务5：报表输出与回读验证

导出3个CSV到输出目录，并重新读取验证。

In [ ]:
# 定义导出
outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# 导出
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    # 回读验证
    reloaded = pd.read_csv(path)
    print(f"{filename}: 导出 shape={table.shape}, 回读 shape={reloaded.shape} ✅")

print(f"\n输出目录：{OUTPUT_DIR}")
print("\n✅ 检查点5通过：三个CSV均已成功导出并回读")

---
## 任务6：结论、限制与建议

In [ ]:
# 结论1
print("=" * 60)
print("结论1：新用户流失风险最高")
print("=" * 60)
print("在 0-1 年生命周期的用户中，流失率约为30%以上，"
      "与 15 年以上老用户（约8%）相比高出约4倍。"
      "这说明当前样本中，用户使用时长与流失存在较强关联，"
      "可能与新用户的产品适应期和留存体验有关；"
      "仍需结合新用户首次操作行为和引导流程数据进一步验证。")
print("对应证据表：segment_analysis.csv")

print()

# 结论2
print("=" * 60)
print("结论2：返现与流失率呈负相关但不构成因果")
print("=" * 60)
print("新用户群体的平均返现金额较高，但流失率也最高。"
      "这表明在当前样本中，返现金额与流失率之间并非简单的线性关系。"
      "可能与新用户对新平台信任度不足有关，返现本身"
      "可能不足以驱动留存；需结合用户首单转化率和满意度"
      "数据进一步验证。")
print("对应证据表：segment_analysis.csv")

print()

# 结论3
print("=" * 60)
print("结论3：城市等级对流失的影响因生命周期而异")
print("=" * 60)
print("在双维交叉分析中，一线城市（CityTier=1）的新用户"
      "流失率高于低线城市同类用户，但差异幅度因分组而异。"
      "这说明城市等级与流失之间的关联可能受到竞争环境、"
      "用户选择余地等因素的影响；需结合竞品数据和用户"
      "地域分布进一步验证。")
print("对应证据表：cross_analysis.csv")

print()

# 分析限制
print("=" * 60)
print("分析限制")
print("=" * 60)
print("1. 数据缺少订单金额和订单日期，无法计算 GMV、客单价或时间趋势；\n"
      "2. CashbackAmount 是返现金额，不能等同于消费金额或销售额；\n"
      "3. 当前数据为横截面数据，只能观察关联，无法确定因果关系；\n"
      "4. 部分双维组合样本量较小（<30），结论需谨慎解读。")

print()

# 运营建议
print("=" * 60)
print("运营建议")
print("=" * 60)
print("建议：针对 0-1 年新用户设计专项留存计划，\n"
      "重点关注首次下单体验和满意度引导。\n\n"
      "验证方式：\n"
      "- 对照实验：对部分新用户在注册后7天内推送引导任务，\n"
      "  对比实验组和对照组的30天留存率；\n"
      "- 需要补充的数据：用户首次操作行为日志、首单转化率、\n"
      "  7日内回访率等过程指标。")

---
## 实验反思

In [ ]:
# 1. 本组最重要的数据发现是什么？它对应哪个指标和分组？
#
# 最重要的发现是：0-1 年新用户流失率远高于其他生命周期群体。
# 对应指标：流失率（Churn.mean）；分组：TenureGroup = "0-1年"。
# 该发现直接指向运营中最需要关注的留存问题。
#
# 2. 本组遇到的最大口径问题是什么？如何解决？
#
# 最大的口径问题是 CashbackAmount 的含义容易误解。
# 该字段是返现金额而非消费金额，不能作为销售额或客单价。
# 解决方式：在分析中明确标注字段含义，结论中使用"返现"而非"消费"。
#
# 3. 哪条结论最容易被误解为因果关系？应如何改写？
#
# 结论1（新用户流失率高）容易被理解为"使用时间短导致流失"。
# 应改写为："在当前样本中，0-1年用户群体的流失率显著高于其他群体，
# 这可能与新用户留存体验有关，但需要更多数据验证是否存在因果关系。"
#
# 4. 如果补充一个字段，你最希望增加什么？它能支持什么新分析？
#
# 最希望增加"订单日期"字段。有了时间信息可以：
# - 计算用户从注册到首单的时间间隔；
# - 分析流失前的行为趋势（如下单频率下降）；
# - 构建时间序列分析，观察月度/季度流失率变化。
#
# 5. 第6天准备把哪两张统计表转化为图表？为什么？
#
# 选择 segment_analysis.csv 和 cross_analysis.csv。
# segment_analysis 适合用柱状图展示各生命周期阶段的流失率和订单数对比；
# cross_analysis 适合用热力图展示 TenureGroup × CityTier 的流失率分布，
# 直观看到哪些组合是高流失区域。